__Task_1__ - Theory

Leave-One-Out - это кросс-валидация, для каждой итерации одна точка используется как валидационная, остальные - для обучения. Плюсы: максимально использует данные обучения, без потерь. Минусы: медленно на big data.

Grid Search - перебирает все комбинации гиперпараметров из заданной сетки.
Randomized Grid Search - случайным образом выбирает подмножество комбинаций из заданной сетки (быстрее).
Bayesian Optimization - строит вероятностную модель зависимости между параметрами и качеством, чтобы выбирать следующие параметры осознанно.

Классификация методов отбора признаков:
Filter methods - оценивают признаки независимо от модели (Pearson, Chi2).
Wrapper methods - используют модель для оценки подмножеств признаков (RFE).
Embedded methods - отбор встроен в обучение модели (Lasso).

Pearson correlation - измеряет линейную зависимость между двумя непрерывными переменными.
Chi2 test - проверяет зависимость категориальных переменных, используется для оценки связи признака и таргета в задачах классификации.

Lasso - добавляет штраф за абсолютные знаечния коэффициентов, в результате чего часть весов обнуляется. Исключает ненужные признаки.

Permutation importance - метод оценки важности признака путем случайной перестановки его значений: если метрика сильно ухудшилась - метод важен.

SHAP - метод интерпритации моделей, основанный на теории корпоративных игр. Оценивает вклад каждого признака в предсказание, обеспечивает глобальную и локальную интерпретацию модели.

__Task_2__ - Preprocessing from previous task

In [1]:
import pandas as pd
import numpy as np
from collections import defaultdict, Counter
from sklearn.utils import check_random_state
from sklearn.model_selection import KFold, GroupKFold, StratifiedKFold, TimeSeriesSplit, cross_val_score, GridSearchCV, RandomizedSearchCV
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.linear_model import Lasso, ElasticNet
from sklearn.metrics import mean_squared_error
from sklearn.inspection import permutation_importance
import shap
import optuna
from scipy.stats import uniform

/home/max/ML.Project_3.ID_1254800-1/daisymal/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
df = pd.read_json("train.json")
print(df.shape)
df.head(2)

(49352, 15)


,bathrooms,bedrooms,building_id,created,description,display_address,features,latitude,listing_id,longitude,manager_id,photos,price,street_address,interest_level
4,1.0,1,8579a0b0d54db803821a35a4a615e97a,2016-06-16 05:55:27,Spacious 1 Bedroom 1 Bathroom in Williamsburg!...,145 Borinquen Place,"[Dining Room, Pre-War, Laundry in Building, Di...",40.7108,7170325,-73.9539,a10db4590843d78c784171a107bdacb4,[https://photos.renthop.com/2/7170325_3bb5ac84...,2400,145 Borinquen Place,medium
6,1.0,2,b8e75fc949a6cd8225b455648a951712,2016-06-01 05:44:33,BRAND NEW GUT RENOVATED TRUE 2 BEDROOMFind you...,East 44th,"[Doorman, Elevator, Laundry in Building, Dishw...",40.7513,7092344,-73.9722,955db33477af4f40004820b4aed804a0,[https://photos.renthop.com/2/7092344_7663c19a...,3800,230 East 44th,low


In [3]:
labels = df["interest_level"]
encoder = LabelEncoder()
df["interest_level_encoded"] = encoder.fit_transform(labels)
label_mapping = dict(zip(encoder.classes_, encoder.transform(encoder.classes_)))
df.drop('interest_level', axis=1, inplace=True)
df.head(2)

,bathrooms,bedrooms,building_id,created,description,display_address,features,latitude,listing_id,longitude,manager_id,photos,price,street_address,interest_level_encoded
4,1.0,1,8579a0b0d54db803821a35a4a615e97a,2016-06-16 05:55:27,Spacious 1 Bedroom 1 Bathroom in Williamsburg!...,145 Borinquen Place,"[Dining Room, Pre-War, Laundry in Building, Di...",40.7108,7170325,-73.9539,a10db4590843d78c784171a107bdacb4,[https://photos.renthop.com/2/7170325_3bb5ac84...,2400,145 Borinquen Place,2
6,1.0,2,b8e75fc949a6cd8225b455648a951712,2016-06-01 05:44:33,BRAND NEW GUT RENOVATED TRUE 2 BEDROOMFind you...,East 44th,"[Doorman, Elevator, Laundry in Building, Dishw...",40.7513,7092344,-73.9722,955db33477af4f40004820b4aed804a0,[https://photos.renthop.com/2/7092344_7663c19a...,3800,230 East 44th,1


In [4]:
df["features_cleaned"] = df["features"].astype(str).str.replace(r"[\[\]'\" ]", '', regex=True)
df["features_list"] = df["features_cleaned"].str.split(',')

In [5]:
all_features = []
for _, row in df.iterrows():
    if isinstance(row["features_list"], list):
        all_features.extend([f for f in row["features_list"] if f.strip() and f.strip() != ":"])

feature_counts = Counter(all_features)
top_20 = feature_counts.most_common(20)

for feature, count in top_20: print(feature)

Elevator
CatsAllowed
HardwoodFloors
DogsAllowed
Doorman
Dishwasher
NoFee
LaundryinBuilding
FitnessCenter
Pre-War
LaundryinUnit
RoofDeck
OutdoorSpace
DiningRoom
HighSpeedInternet
Balcony
SwimmingPool
LaundryInBuilding
NewConstruction
Terrace


In [6]:
top_20 = [f for f, _ in Counter(all_features).most_common(20)]
df_features_20 = df.copy()

for feature in top_20:
    df_features_20[feature] = df_features_20["features_list"].apply(
        lambda x: int(feature in x) if isinstance(x, list) else 0
    )

print(df_features_20.shape)

(49352, 37)


In [7]:
df_features_20.rename(columns={"interest_level_encoded" : "interest_level"}, inplace=True)
df_features_20.drop(['features', 'features_cleaned', 'features_list'], axis=1, inplace=True)

X = df_features_20.drop("price", axis=1)
X['created'] = pd.to_datetime(X['created'])
print(f"X shape: {X.shape}")
y = df_features_20["price"]
print(f"y shape: {y.shape}")

X shape: (49352, 33)
y shape: (49352,)


__Task_3__ - splitting functions with checking

In [8]:
def split_data_random_2(X, y, test_size=0.2, seed=42):
    np.random.seed(seed)
    indices = np.arange(len(X))
    np.random.shuffle(indices)
    test_len = int(len(X) * test_size)
    test_idx = indices[:test_len]
    train_idx = indices[test_len:]
    X_train = X.iloc[train_idx].reset_index(drop=True)
    y_train = y.iloc[train_idx].reset_index(drop=True)
    X_test = X.iloc[test_idx].reset_index(drop=True)
    y_test = y.iloc[test_idx].reset_index(drop=True)
    return X_train, y_train, X_test, y_test

def split_data_random_3(X, y, validation_size=0.2, test_size=0.2, seed=42):
    np.random.seed(seed)
    indices = np.arange(len(X))
    np.random.shuffle(indices)
    total_len = len(X)
    test_len = int(total_len * test_size)
    val_len = int(total_len * validation_size)
    train_len = total_len - test_len - val_len
    train_idx = indices[:train_len]
    val_idx = indices[train_len:train_len + val_len]
    test_idx = indices[train_len + val_len:]
    X_train = X.iloc[train_idx].reset_index(drop=True)
    y_train = y.iloc[train_idx].reset_index(drop=True)
    X_val = X.iloc[val_idx].reset_index(drop=True)
    y_val = y.iloc[val_idx].reset_index(drop=True)
    X_test = X.iloc[test_idx].reset_index(drop=True)
    y_test = y.iloc[test_idx].reset_index(drop=True)
    return X_train, y_train, X_val, y_val, X_test, y_test

def split_data_by_date_2(X, y, date_split):
    X['created'] = pd.to_datetime(X['created'])
    date_split = pd.to_datetime(date_split)
    train_mask = X['created'] < date_split
    test_mask = X['created'] >= date_split
    X_train = X[train_mask]
    y_train = y[train_mask]
    X_test = X[test_mask]
    y_test = y[test_mask]
    return X_train, y_train, X_test, y_test

def split_data_by_date_3(X, y, validation_date, test_date):
    X['created'] = pd.to_datetime(X['created'])
    validation_date = pd.to_datetime(validation_date)
    test_date = pd.to_datetime(test_date)
    train_mask = X['created'] < validation_date
    val_mask = (X['created'] >= validation_date) & (X['created'] < test_date)
    test_mask = X['created'] >= test_date
    X_train = X[train_mask]
    y_train = y[train_mask]
    X_val = X[val_mask]
    y_val = y[val_mask]
    X_test = X[test_mask]
    y_test = y[test_mask]
    return X_train, y_train, X_val, y_val, X_test, y_test

In [9]:
X_train, y_train, X_test, y_test = split_data_random_2(X, y)

print(f"X_train: {X_train.shape} - {round(X_train.shape[0]/df_features_20.shape[0], 3)*100}%")
print(f"X_test: {X_test.shape} - {round(X_test.shape[0]/df_features_20.shape[0], 3)*100}%")
print(f"y_train: {y_train.shape}")
print(f"y_test: {y_test.shape}")

X_train: (39482, 33) - 80.0%
X_test: (9870, 33) - 20.0%
y_train: (39482,)
y_test: (9870,)


In [10]:
X_train, y_train, X_val, y_val, X_test, y_test = split_data_random_3(X, y)

print(f"X_train: {X_train.shape} - {round(X_train.shape[0]/df_features_20.shape[0], 3)*100}%") 
print(f"X_val: {X_val.shape} - {round(X_val.shape[0]/df_features_20.shape[0], 3)*100}%")
print(f"X_test: {X_test.shape} - {round(X_test.shape[0]/df_features_20.shape[0], 3)*100}%")
print(f"y_train: {y_train.shape}")
print(f"y_val: {y_val.shape}")
print(f"y_test: {y_test.shape}")

X_train: (29612, 33) - 60.0%
X_val: (9870, 33) - 20.0%
X_test: (9870, 33) - 20.0%
y_train: (29612,)
y_val: (9870,)
y_test: (9870,)


In [11]:
X_train, y_train, X_test, y_test = split_data_by_date_2(X, y, "2016-06-12")

print(f"X_train: {X_train.shape} - {round(X_train.shape[0]/X.shape[0], 3)*100}%") 
print(f"X_test: {X_test.shape} - {round(X_test.shape[0]/X.shape[0], 3)*100}%")
print(f"y_train: {y_train.shape}")
print(f"y_test: {y_test.shape}")
print(max(X_train['created']) < max(X_test['created']))

X_train: (39075, 33) - 79.2%
X_test: (10277, 33) - 20.8%
y_train: (39075,)
y_test: (10277,)
True


In [12]:
X_train, y_train, X_val, y_val, X_test, y_test = split_data_by_date_3(X, y, "2016-05-25", "2016-06-12")

print(f"X_train: {X_train.shape} - {round(X_train.shape[0]/X.shape[0], 3)*100}%")
print(f"X_val: {X_val.shape} - {round(X_val.shape[0]/X.shape[0], 3)*100}%")
print(f"X_test: {X_test.shape} - {round(X_test.shape[0]/X.shape[0], 3)*100}%")
print(f"y_train: {y_train.shape}")
print(f"y_val: {y_val.shape}")
print(f"y_test: {y_test.shape}")
print(max(X_train['created']) < max(X_val['created']) < max(X_test['created']))

X_train: (29610, 33) - 60.0%
X_val: (9465, 33) - 19.2%
X_test: (10277, 33) - 20.8%
y_train: (29610,)
y_val: (9465,)
y_test: (10277,)
True


__Task_4__ - crossvalidation methods with checking

In [13]:
def k_fold_split(X, k, seed=42):
    rng = np.random.RandomState(seed)
    indices = np.arange(len(X))
    rng.shuffle(indices)
    fold_sizes = np.full(k, len(X) // k)
    fold_sizes[:len(X) % k] += 1
    current, folds = 0, []
    for fold_size in fold_sizes:
        test_idx = np.sort(indices[current:current + fold_size])
        train_idx = np.sort(np.setdiff1d(indices, test_idx, assume_unique=True))
        folds.append((train_idx, test_idx))
        current += fold_size
    return folds


def grouped_k_fold_split(X, k, group_field, seed=42):
    np.random.seed(seed)
    groups = X[group_field]
    group_to_indices = defaultdict(list)
    for idx, g in enumerate(groups):
        group_to_indices[g].append(idx)
    sorted_groups = sorted(group_to_indices.items(), key=lambda x: -len(x[1]))
    folds = [[] for _ in range(k)]
    fold_sizes = [0] * k
    for _, indices in sorted_groups:
        min_fold = fold_sizes.index(min(fold_sizes))
        folds[min_fold].extend(indices)
        fold_sizes[min_fold] += len(indices)
    final_folds = []
    for i in range(k):
        test_idx = np.array(folds[i])
        train_idx = np.setdiff1d(np.arange(len(X)), test_idx)
        final_folds.append((train_idx, test_idx))
    return final_folds


def stratified_k_fold_split(X, y, k, stratify_field, random_state=42):
    rng = check_random_state(random_state)
    stratify_values = X[stratify_field].values
    indices = np.arange(len(X))
    shuffled_indices = rng.permutation(indices)
    stratify_values = stratify_values[shuffled_indices]
    _, y_encoded = np.unique(stratify_values, return_inverse=True)
    fold_sizes = np.bincount(y_encoded)
    cls_counts = np.zeros((np.max(y_encoded) + 1, k), dtype=int)
    for cls in np.unique(y_encoded):
        cls_mask = y_encoded == cls
        cls_indices = np.where(cls_mask)[0]
        fold_counts = np.full(k, len(cls_indices) // k)
        fold_counts[:len(cls_indices) % k] += 1
        cls_counts[cls] = fold_counts
    fold_indices = [[] for _ in range(k)]
    current_per_class = {cls: 0 for cls in np.unique(y_encoded)}
    for cls in np.unique(y_encoded):
        cls_idx = np.where(y_encoded == cls)[0]
        for fold in range(k):
            start = current_per_class[cls]
            stop = start + cls_counts[cls][fold]
            fold_indices[fold].extend(cls_idx[start:stop])
            current_per_class[cls] = stop
    folds = []
    for i in range(k):
        test_mask = np.zeros(len(X), dtype=bool)
        test_mask[shuffled_indices[fold_indices[i]]] = True
        test_idx = np.where(test_mask)[0]
        train_idx = np.where(~test_mask)[0]
        folds.append((train_idx, test_idx))
    return folds


def time_series_split(X, k=5, date_field=None, max_train_size=None):
    n_samples = len(X)
    sorted_indices = np.argsort(X[date_field].values) if date_field else np.arange(n_samples)
    test_size = n_samples // (k + 1)
    remainder = n_samples - test_size * k
    splits = []
    for i in range(k):
        test_start = remainder + i * test_size
        test_end = test_start + test_size
        if max_train_size is not None: train_start = max(0, test_start - max_train_size)
        else: train_start = 0
        train_idx = sorted_indices[train_start:test_start]
        test_idx = sorted_indices[test_start:test_end]
        splits.append((train_idx, test_idx))
    return splits

In [14]:
kf_indices = k_fold_split(X, 5, seed=42)
for row in kf_indices:
    print(row)

(array([    0,     2,     3, ..., 49349, 49350, 49351], shape=(39481,)), array([    1,     4,     6, ..., 49339, 49340, 49341], shape=(9871,)))
(array([    1,     2,     3, ..., 49349, 49350, 49351], shape=(39481,)), array([    0,    11,    13, ..., 49342, 49344, 49345], shape=(9871,)))
(array([    0,     1,     2, ..., 49347, 49348, 49349], shape=(39482,)), array([    3,     8,    14, ..., 49343, 49350, 49351], shape=(9870,)))
(array([    0,     1,     2, ..., 49349, 49350, 49351], shape=(39482,)), array([    5,    15,    18, ..., 49346, 49347, 49348], shape=(9870,)))
(array([    0,     1,     3, ..., 49348, 49350, 49351], shape=(39482,)), array([    2,     9,    10, ..., 49333, 49337, 49349], shape=(9870,)))


In [15]:
gkf_indices = grouped_k_fold_split(X, k=5, group_field="bedrooms")
for row in gkf_indices:
    print(row)

(array([    1,     2,     3, ..., 49348, 49350, 49351], shape=(33600,)), array([    0,     8,    13, ..., 49344, 49345, 49349], shape=(15752,)))
(array([    0,     3,     4, ..., 49347, 49349, 49351], shape=(34729,)), array([    1,     2,    11, ..., 49343, 49348, 49350], shape=(14623,)))
(array([    0,     1,     2, ..., 49349, 49350, 49351], shape=(39877,)), array([    4,     7,    10, ..., 49335, 49339, 49346], shape=(9475,)))
(array([    0,     1,     2, ..., 49348, 49349, 49350], shape=(42076,)), array([    3,     5,     6, ..., 49342, 49347, 49351], shape=(7276,)))
(array([    0,     1,     2, ..., 49349, 49350, 49351], shape=(47126,)), array([   32,    47,    75, ..., 11386, 10672, 41666], shape=(2226,)))


In [16]:
skf_indices = stratified_k_fold_split(X, y=X["interest_level"], k=5, stratify_field="interest_level", random_state=42)
for row in skf_indices:
    print(row)

for fold, (train_idx, test_idx) in enumerate(skf_indices, 1):
    y_train = X.iloc[train_idx]["interest_level"]
    y_test = X.iloc[test_idx]["interest_level"]

    train_dist = y_train.value_counts(normalize=True).sort_index()
    test_dist = y_test.value_counts(normalize=True).sort_index()

    print(f"\nFold {fold}")
    print("Train class distribution:")
    for cls, pct in train_dist.items():
        print(f"  {cls}: {round(pct * 100, 2)}%")

    print("Test class distribution:")
    for cls, pct in test_dist.items():
        print(f"  {cls}: {round(pct * 100, 2)}%")

(array([    0,     2,     3, ..., 49349, 49350, 49351], shape=(39481,)), array([    1,     4,     6, ..., 49339, 49340, 49341], shape=(9871,)))
(array([    1,     2,     3, ..., 49349, 49350, 49351], shape=(39481,)), array([    0,    11,    13, ..., 49342, 49344, 49345], shape=(9871,)))
(array([    0,     1,     2, ..., 49347, 49348, 49349], shape=(39481,)), array([    3,     8,    14, ..., 49343, 49350, 49351], shape=(9871,)))
(array([    0,     1,     2, ..., 49349, 49350, 49351], shape=(39481,)), array([    5,    15,    18, ..., 49346, 49347, 49348], shape=(9871,)))
(array([    0,     1,     3, ..., 49348, 49350, 49351], shape=(39484,)), array([    2,     9,    10, ..., 49333, 49337, 49349], shape=(9868,)))

Fold 1
Train class distribution:
  0: 7.78%
  1: 69.47%
  2: 22.75%
Test class distribution:
  0: 7.78%
  1: 69.47%
  2: 22.75%

Fold 2
Train class distribution:
  0: 7.78%
  1: 69.47%
  2: 22.75%
Test class distribution:
  0: 7.78%
  1: 69.47%
  2: 22.75%

Fold 3
Train class di

In [17]:
X_sorted = X.sort_values("created").reset_index(drop=True)

tss_indices = time_series_split(X_sorted, k=5, date_field="created")
for row in tss_indices:
    print(row)

(array([   0,    1,    2, ..., 8224, 8225, 8226], shape=(8227,)), array([ 8227,  8228,  8229, ..., 16449, 16450, 16451], shape=(8225,)))
(array([    0,     1,     2, ..., 16449, 16450, 16451], shape=(16452,)), array([16452, 16453, 16454, ..., 24674, 24675, 24676], shape=(8225,)))
(array([    0,     1,     2, ..., 24674, 24675, 24676], shape=(24677,)), array([24677, 24678, 24679, ..., 32899, 32900, 32901], shape=(8225,)))
(array([    0,     1,     2, ..., 32899, 32900, 32901], shape=(32902,)), array([32902, 32903, 32904, ..., 41124, 41125, 41126], shape=(8225,)))
(array([    0,     1,     2, ..., 41124, 41125, 41126], shape=(41127,)), array([41127, 41128, 41129, ..., 49349, 49350, 49351], shape=(8225,)))


__Task_5__ - comparing with original sklearn methods

In [18]:
kf_skl = KFold(n_splits=5, shuffle=True, random_state=42)
kf_skl_indices = list(kf_skl.split(X))

equal = all(
    np.array_equal(np.sort(a1), np.sort(a2)) and np.array_equal(np.sort(b1), np.sort(b2))
    for (a1, b1), (a2, b2) in zip(kf_indices, kf_skl_indices)
)

print(equal)

True


In [19]:
gkf_skl = GroupKFold(n_splits=5)
gkf_skl_indices = list(gkf_skl.split(X, y=None, groups=X['bedrooms']))

equal = all(
    np.array_equal(np.sort(a1), np.sort(a2)) and np.array_equal(np.sort(b1), np.sort(b2))
    for (a1, b1), (a2, b2) in zip(gkf_indices, gkf_skl_indices)
)

print(equal)

True


In [20]:
skf_indices = stratified_k_fold_split(X, y=X["interest_level"], k=5, stratify_field="interest_level", random_state=42)

skf_skl = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
skf_skl_indices = list(skf_skl.split(X, y=X["interest_level"]))

equal = all(
    np.array_equal(np.sort(train1), np.sort(train2)) and
    np.array_equal(np.sort(test1), np.sort(test2))
    for (train1, test1), (train2, test2) in zip(kf_indices, kf_skl_indices)
)

print(equal)

True


In [21]:
tss_skl = TimeSeriesSplit(n_splits=5)
tss_skl_indices = list(tss_skl.split(X_sorted))

equal = all(
    np.array_equal(np.sort(a1), np.sort(a2)) and np.array_equal(np.sort(b1), np.sort(b2))
    for (a1, b1), (a2, b2) in zip(tss_indices, tss_skl_indices)
)

print(equal)

True


__Task_6__ - Feature selection

In [22]:
X_train, y_train, X_val, y_val, X_test, y_test = split_data_by_date_3(X, y, "2016-05-25", "2016-06-12 08:00:00")

print(f"X_train: {X_train.shape} - {round(X_train.shape[0]/df_features_20.shape[0], 3)*100}%") 
print(f"X_val: {X_val.shape} - {round(X_val.shape[0]/df_features_20.shape[0], 3)*100}%")
print(f"X_test: {X_test.shape} - {round(X_test.shape[0]/df_features_20.shape[0], 3)*100}%")
print(f"y_train: {y_train.shape}")
print(f"y_val: {y_val.shape}")
print(f"y_test: {y_test.shape}")

X_train: (29610, 33) - 60.0%
X_val: (9847, 33) - 20.0%
X_test: (9895, 33) - 20.0%
y_train: (29610,)
y_val: (9847,)
y_test: (9895,)


In [23]:
scaler = StandardScaler()
numeric_columns = X_train.select_dtypes(include=np.number).columns
X_train_scaled = scaler.fit_transform(X_train[numeric_columns])
X_val_scaled = scaler.transform(X_val[numeric_columns])
X_test_scaled = scaler.transform(X_test[numeric_columns])

In [24]:
lasso = Lasso()
lasso.fit(X_train_scaled, y_train)
rmse_all_features = np.sqrt(mean_squared_error(y_val, lasso.predict(X_val_scaled)))
print(f"All features: {round(rmse_all_features, 2)}")

All features: 2005.58


In [25]:
coef = pd.Series(lasso.coef_, index=numeric_columns)
top10_coef_features = coef.abs().sort_values(ascending=False).head(10).index

for i in range(len(top10_coef_features[0])):
    print(top10_coef_features[i])

bathrooms
Doorman
longitude
latitude
bedrooms
LaundryinUnit
LaundryinBuilding
DogsAllowed
NoFee


In [26]:
X_train_top10 = X_train[top10_coef_features]
X_val_top10 = X_val[top10_coef_features]
X_test_top10 = X_test[top10_coef_features]

X_train_top10_scaled = scaler.fit_transform(X_train_top10)
X_val_top10_scaled = scaler.transform(X_val_top10)
X_test_top10_scaled = scaler.transform(X_test_top10)

In [27]:
lasso.fit(X_train_top10_scaled, y_train)
rmse_top10 = np.sqrt(mean_squared_error(y_val, lasso.predict(X_val_top10_scaled)))
print(f"10 top features: {round(rmse_top10, 2)}")

10 top features: 1998.65


In [28]:
nan_ratio = X.isna().mean()
X_filtered = X.loc[:, nan_ratio <= 0.3]
X_filtered_numeric = X_filtered.select_dtypes(include=[np.number])

corrs = X_filtered_numeric.corrwith(y).abs()
top10_corr = corrs.sort_values(ascending=False).head(10).index

X_train_corr = X_train[top10_corr]
X_val_corr = X_val[top10_corr]
X_test_corr = X_test[top10_corr]

X_train_corr_scaled = scaler.fit_transform(X_train_corr)
X_val_corr_scaled = scaler.transform(X_val_corr)
X_test_corr_scaled = scaler.transform(X_test_corr)

lasso.fit(X_train_corr_scaled, y_train)
rmse_nan_n_corr = np.sqrt(mean_squared_error(y_val, lasso.predict(X_val_corr_scaled)))

print(f"NAN and corr: {round(rmse_nan_n_corr, 2)}")

NAN and corr: 2011.65


In [29]:
lasso = Lasso()
lasso.fit(X_train_scaled, y_train)

,alpha,1.0
,fit_intercept,True
,precompute,False
,copy_X,True
,max_iter,1000
,tol,0.0001
,warm_start,False
,positive,False
,random_state,None
,selection,'cyclic'


In [30]:
result = permutation_importance(
    lasso,
    X_val_scaled,
    y_val,
    n_repeats=10,
    random_state=42,
    scoring='neg_mean_absolute_percentage_error'
)

In [31]:
X_val_numeric = X_val.select_dtypes(include=[np.number])
importance_scores = pd.Series(result.importances_mean, index=X_val_numeric.columns)
top10_perm = importance_scores.sort_values(ascending=False).head(10).index

In [32]:
print(top10_perm)

Index(['bathrooms', 'Doorman', 'bedrooms', 'LaundryinUnit', 'longitude',
       'latitude', 'Elevator', 'Terrace', 'DogsAllowed', 'CatsAllowed'],
      dtype='object')


In [33]:
X_train_scaled_df = pd.DataFrame(X_train_scaled, columns=X_val_numeric.columns)
X_val_scaled_df = pd.DataFrame(X_val_scaled, columns=X_val_numeric.columns)
X_test_scaled_df = pd.DataFrame(X_test_scaled, columns=X_val_numeric.columns)

X_train_top10 = X_train_scaled_df[top10_perm]
X_val_top10 = X_val_scaled_df[top10_perm]
X_test_top10 = X_test_scaled_df[top10_perm]

In [34]:
importances_mean = pd.Series(result.importances_mean, index=X_val_scaled_df.columns).round(5)
importances_mean_sorted = importances_mean.sort_values(ascending=False)
importances_mean_sorted.head(10)

bathrooms        0.13484
Doorman          0.07228
bedrooms         0.05770
LaundryinUnit    0.01265
longitude        0.00854
latitude         0.00835
Elevator         0.00488
Terrace          0.00335
DogsAllowed      0.00315
CatsAllowed      0.00240
dtype: float64

In [35]:
lasso = Lasso()
lasso.fit(X_train_top10, y_train)

,alpha,1.0
,fit_intercept,True
,precompute,False
,copy_X,True
,max_iter,1000
,tol,0.0001
,warm_start,False
,positive,False
,random_state,None
,selection,'cyclic'


In [36]:
rmse_perm = np.sqrt(mean_squared_error(y_val, lasso.predict(X_val_top10)))
print(f"Permutation rmse: {round(rmse_perm, 2)}")

Permutation rmse: 2008.71


In [37]:
lasso.fit(X_train_top10, y_train)

explainer = shap.Explainer(lasso.predict, X_val_top10)
shap_values = explainer(X_val_top10)

ExactExplainer explainer: 9848it [01:00, 146.98it/s]                          


In [38]:
mean_abs_shap = np.abs(shap_values.values).mean(axis=0)
shap_df = pd.Series(mean_abs_shap, index=X_val_top10.columns)

In [39]:
top10_shap = shap_df.sort_values(ascending=False).head(10).index
print(top10_shap)

Index(['bathrooms', 'Doorman', 'bedrooms', 'DogsAllowed', 'LaundryinUnit',
       'CatsAllowed', 'Elevator', 'latitude', 'Terrace', 'longitude'],
      dtype='object')


In [40]:
X_train_shap = X_train_scaled_df[top10_shap]
X_val_shap = X_val_scaled_df[top10_shap]
X_test_shap = X_test_scaled_df[top10_shap]

lasso = Lasso()
lasso.fit(X_train_shap, y_train)

rmse_shap = np.sqrt(mean_squared_error(y_val, lasso.predict(X_val_shap)))
print(f"Shap rmse: {round(rmse_shap, 2)}")

Shap rmse: 2008.71


In [41]:
results = [
    {
        "Method": "All features (baseline)",
        "Amount of features": 33,
        "rmse validation": round(rmse_all_features, 2),
    },
    {
        "Method": "Permutation Importance top10",
        "Amount of features": 10,
        "rmse validation": round(rmse_perm, 2),
    },
    {
        "Method": "SHAP Importance top10",
        "Amount of features": 10,
        "rmse validation": round(rmse_shap, 2),
    },
    {
        "Method": "Top10 features",
        "Amount of features": 10,
        "rmse validation": round(rmse_top10, 2),
    },
    {
        "Method": "Nan and corr top10",
        "Amount of features": 10,
        "rmse validation": round(rmse_nan_n_corr, 2),
    },
]

df_results = pd.DataFrame(results)
display(df_results.set_index("Method"))


,Amount of features,rmse validation
Method,,
All features (baseline),33,2005.58
Permutation Importance top10,10,2008.71
SHAP Importance top10,10,2008.71
Top10 features,10,1998.65
Nan and corr top10,10,2011.65


__Task_7__ - Hyperparameters research

In [42]:
param_grid = {
    'alpha': [0.001, 0.01, 0.1, 1, 10],
    'l1_ratio': [0.1, 0.5, 0.7, 0.9, 1]
}

elastic = ElasticNet(max_iter=10000)

grid_search = GridSearchCV(
    elastic,
    param_grid,
    scoring='neg_root_mean_squared_error',
    cv=5,
    n_jobs=-1
)

grid_search.fit(X_train_scaled, y_train)

print("Best params (GridSearch):", grid_search.best_params_)
print("Best RMSE:", -grid_search.best_score_)

Best params (GridSearch): {'alpha': 1, 'l1_ratio': 0.9}
Best RMSE: 5479.386514307573


In [43]:
param_dist = {
    'alpha': uniform(0.001, 10),
    'l1_ratio': uniform(0, 1)
}

random_search = RandomizedSearchCV(
    elastic,
    param_distributions=param_dist,
    n_iter=50,
    scoring='neg_root_mean_squared_error',
    cv=5,
    n_jobs=-1,
    random_state=42
)

random_search.fit(X_train_scaled, y_train)

print("Best params (RandomSearch):", random_search.best_params_)
print("Best RMSE:", -random_search.best_score_)


Best params (RandomSearch): {'alpha': np.float64(0.5818361216819946), 'l1_ratio': np.float64(0.8661761457749352)}
Best RMSE: 5479.510215531298


In [44]:
def objective(trial):
    alpha = trial.suggest_float('alpha', 0.001, 10.0, log=True)
    l1_ratio = trial.suggest_float('l1_ratio', 0.0, 1.0)

    model = ElasticNet(alpha=alpha, l1_ratio=l1_ratio, max_iter=10000)
    rmse = -cross_val_score(model, X_train_scaled, y_train,
                            scoring='neg_root_mean_squared_error', cv=5, n_jobs=-1).mean()
    return rmse

study = optuna.create_study(direction='minimize')
study.optimize(objective, n_trials=50)

print("Best params (Optuna):", study.best_params)
print("Best RMSE:", study.best_value)

[I 2025-07-13 02:55:26,136] A new study created in memory with name: no-name-cb314fc4-d2b8-4ad3-a1a2-544ba82867fd
[I 2025-07-13 02:55:27,008] Trial 0 finished with value: 5483.002941443194 and parameters: {'alpha': 0.0012419566444655367, 'l1_ratio': 0.22008844437493058}. Best is trial 0 with value: 5483.002941443194.
[I 2025-07-13 02:55:27,679] Trial 1 finished with value: 5482.7914338882765 and parameters: {'alpha': 0.005389925542694188, 'l1_ratio': 0.6403739657275903}. Best is trial 1 with value: 5482.7914338882765.
[I 2025-07-13 02:55:27,860] Trial 2 finished with value: 5522.380152696463 and parameters: {'alpha': 5.808369365184024, 'l1_ratio': 0.8934467157061938}. Best is trial 1 with value: 5482.7914338882765.
[I 2025-07-13 02:55:28,062] Trial 3 finished with value: 5479.777106800155 and parameters: {'alpha': 0.2952898062778065, 'l1_ratio': 0.5904654341777967}. Best is trial 3 with value: 5479.777106800155.
[I 2025-07-13 02:55:28,294] Trial 4 finished with value: 5479.533805699809

Best params (Optuna): {'alpha': 0.7715130920362969, 'l1_ratio': 0.8742735827670567}
Best RMSE: 5479.412216302248


In [45]:
results = {
    "GridSearch": -grid_search.best_score_,
    "RandomSearch": -random_search.best_score_,
    "Optuna": study.best_value
}

for method, rmse in results.items():
    print(f"{method}: RMSE = {rmse:.4f}")

GridSearch: RMSE = 5479.3865
RandomSearch: RMSE = 5479.5102
Optuna: RMSE = 5479.4122
